[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/sandbox/pharmit_hits_to_molport.ipynb)

# From Pharmit hits to a non-redundant MolPort list

**Sandbox notebook - not workshop material.**

Takes the SDF files Pharmit returns for the CpABC1 pharmacophore (see `pharmaconet_to_pharmit.ipynb`)
and turns them into a non-redundant compound list: every matching compound once, with close
analogues removed and the best-fitting member of each family kept. It writes two files:

- `pharmit_hits_molport_nonredundant.csv` - one column of **SMILES** (a molecule written as a line
  of text) and one of **MolPort IDs** (the catalogue number you order the compound with).
- `pharmit_hits_nonredundant_smiles.csv` - the same compounds with only the SMILES column, the input
  for the Ersilia Model Hub.

Pharmit's SDF files are far from a compound list. Each compound shows up many times, as different 3D
**conformers** (shapes of the same molecule) and sometimes as different **stereoisomers** (mirror-image
or otherwise differently arranged versions). This notebook collapses all of that, step by step.

## What you will do

- Read every pose from the Pharmit SDF files, with its MolPort IDs and fit score
- Collapse conformers and stereoisomers so each MolPort compound appears once
- Merge different forms of the same molecule (salts, charges, tautomers)
- Remove close analogues, keeping the compound that best fits the pharmacophore
- Save both tables as CSV

## 1. Find the SDF files

The Pharmit results are large (the biggest is 140 MB), so they are not in the repository. Locally
they live in `bigfiles/` at the repository root, which git ignores. Any file named
`provisional_pharmit_query_results_*.sdf` there is read. The two CSV files are written next to them.

In Colab, you are asked to upload the SDF files instead, and the CSVs are written to the Colab
folder (download them from the file panel on the left before the runtime disconnects).

> **Note:** RDKit, the chemistry library used throughout, is not preinstalled in Colab, so the cell
> installs it there. Locally, use a kernel that already has it (e.g. `ubcedd312`).

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit"], check=True)
    from google.colab import files
    print("Upload the provisional_pharmit_query_results_*.sdf files")
    files.upload()
# Locally the notebook sits in sandbox/, so the repo root is one level up.
BIG = Path.cwd() if IN_COLAB else Path.cwd().parent / "bigfiles"

SDF_FILES = sorted(BIG.glob("provisional_pharmit_query_results_*.sdf"))
for path in SDF_FILES:
    print(f"{path.name}: {path.stat().st_size / 1e6:.0f} MB")

## 2. Read every pose

An SDF file is a list of molecule records, each with 3D coordinates. Pharmit writes one record per
**pose**: a conformer of a compound that fits the pharmacophore. Two parts of each record matter here:

- The **title line** lists every catalogue the compound is sold in, for example
  `MolPort-016-672-966 12133290 MCULE-1421397642 ZINC000012133290`. We keep only the MolPort IDs.
  The spelling varies (`MolPort-` and `Molport-`), and some compounds have more than one ID.
- The **`rmsd`** field says how well the pose fits the pharmacophore, in angstroms. Lower is better.

Let's look at the first title line before reading everything.

In [ ]:
with open(SDF_FILES[0]) as fh:
    print(fh.readline().strip())

Now read all the records. For each one we:

1. Assign stereochemistry from the 3D coordinates, so that the SMILES records which mirror-image
   form this pose is.
2. Write it as a **canonical SMILES**: RDKit always writes the same molecule the same way, so two
   conformers of one compound give identical text.
3. Pull the MolPort IDs out of the title with a regular expression (a text pattern) and write them
   all as `MolPort-###-###-###`.

This takes a minute or two.

In [ ]:
import re
import pandas as pd
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")
ID_RE = re.compile(r"(?i)molport-(\d{3}-\d{3}-\d{3})")

records = []
for path in SDF_FILES:
    with open(path, "rb") as fh:
        for mol in Chem.ForwardSDMolSupplier(fh):
            Chem.AssignStereochemistryFrom3D(mol)
            ids = sorted(f"MolPort-{m}" for m in ID_RE.findall(mol.GetProp("_Name")))
            records.append((Chem.MolToSmiles(mol), ids, float(mol.GetProp("rmsd"))))

poses = pd.DataFrame(records, columns=["smiles", "molport_ids", "rmsd"])
print(f"{len(poses)} poses, {(poses.molport_ids.str.len() == 0).sum()} without a MolPort ID")
poses.head()

## 3. One row per structure

Conformers of the same compound now share a SMILES, so grouping by SMILES collapses them. For each
structure we keep:

- **every MolPort ID** seen for it, across all its poses, and
- its **best (lowest) `rmsd`**, which we use in section 6.

When a structure is sold under more than one MolPort ID, the table keeps one: the first in sorted
order. They are the same molecule, so any of them can be ordered.

In [ ]:
structures = poses.groupby("smiles").agg(
    molport_ids=("molport_ids", lambda col: sorted(set().union(*col))),
    rmsd=("rmsd", "min"),
).reset_index()
structures["molport_id"] = structures.molport_ids.str[0]

print(f"{len(structures)} structures from {len(poses)} poses")
print(f"{(structures.molport_ids.str.len() > 1).sum()} structures are sold under more than one MolPort ID")

## 4. One row per MolPort compound

Some MolPort IDs still appear on more than one row. When a vendor does not specify a compound's
stereochemistry, Pharmit builds every stereoisomer and screens each one, so one product turns into
several SMILES. Here is an example:

In [ ]:
per_id = structures.molport_id.value_counts()
print(f"{(per_id > 1).sum()} MolPort IDs have more than one stereoisomer")
structures[structures.molport_id == per_id[per_id == 2].index[0]][["smiles", "molport_id", "rmsd"]]

The `@` and `@@` in the SMILES mark the two mirror-image forms. Buying the MolPort product gets
you whatever the vendor has, so we keep **one stereoisomer at random** for each ID. The random seed
is fixed, so running the notebook again picks the same ones.

In [ ]:
hits = (structures.sort_values(["molport_id", "smiles"])
        .sample(frac=1, random_state=42)
        .drop_duplicates("molport_id")
        .sort_values("molport_id"))

print(f"{len(hits)} compounds, each MolPort ID and each SMILES once: "
      f"{hits.molport_id.is_unique and hits.smiles.is_unique}")

## 5. Merge forms of the same molecule

Some compounds are still the same molecule written in different ways, so their SMILES do not match:

- **Salts**: a compound sold with a counter-ion, such as a hydrochloride, carries an extra fragment.
- **Charges**: an amine can be drawn protonated (`[NH3+]`) or neutral (`N`).
- **Tautomers**: some molecules can shift a hydrogen from one atom to another. These are two
  drawings of one compound that interconvert in water.

RDKit's `MolStandardize` fixes all three: keep the largest fragment, neutralise it, then pick one
canonical tautomer. We use the result only to spot duplicates, and the table keeps the original
SMILES.

This is only a small fraction of compounds, but these are true duplicates. First we sort the table
by `rmsd`, so that when two rows are the same molecule, the better-fitting one is kept.

> **Note:** tautomer enumeration is slow. This cell takes about 5-6 minutes.

In [ ]:
from rdkit.Chem.MolStandardize import rdMolStandardize

uncharger, tautomers = rdMolStandardize.Uncharger(), rdMolStandardize.TautomerEnumerator()
tautomers.SetMaxTautomers(50)

def standardise(smiles):
    """Return one SMILES shared by all salt, charge and tautomer forms of a molecule."""
    mol = uncharger.uncharge(rdMolStandardize.FragmentParent(Chem.MolFromSmiles(smiles)))
    return Chem.MolToSmiles(tautomers.Canonicalize(mol))

ranked = hits.sort_values(["rmsd", "molport_id"]).reset_index(drop=True)
ranked["standard"] = ranked.smiles.map(standardise)
unique = ranked.drop_duplicates("standard").reset_index(drop=True)
print(f"{len(unique)} distinct molecules ({len(ranked) - len(unique)} merged)")

## 6. Remove close analogues

Many hits are small variations on one scaffold: a methyl swapped for a chlorine, a ring nitrogen
moved one position. Ordering all of them tells you little more than ordering one. We keep one
compound per family.

**How similarity is measured.** A **Morgan fingerprint** records which small atom neighbourhoods a
molecule contains, as a string of 2048 on/off bits. The **Tanimoto similarity** of two fingerprints is
the share of on-bits they have in common: 1 means identical fingerprints and 0 means nothing in
common. Pairs above about 0.7 are usually close analogues.

**How one compound per family is kept.** RDKit's `LeaderPicker` goes down the list in order. A compound
becomes a *leader* unless it has Tanimoto 0.7 or more to a leader already chosen, in which case it is
dropped. Because the list is sorted by `rmsd`, each family is represented by its best-fitting member.

> **Exercise:** change `SIMILARITY` and rerun this section. On this data, 0.8 keeps about 39k compounds,
> 0.6 about 19k and 0.5 about 12k.

In [ ]:
from rdkit.Chem import rdFingerprintGenerator

morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fingerprints = [morgan.GetFingerprint(Chem.MolFromSmiles(s)) for s in unique.smiles]
print(f"{len(fingerprints)} fingerprints")

Run the picker. `LeaderPicker` takes a distance threshold, which is 1 minus the similarity. This
takes about a minute and a half.

In [ ]:
from rdkit.SimDivFilters import rdSimDivPickers

SIMILARITY = 0.7
picks = rdSimDivPickers.LeaderPicker().LazyBitVectorPick(
    fingerprints, len(fingerprints), 1 - SIMILARITY, numThreads=8)
leaders = unique.loc[sorted(picks)]

print(f"{len(leaders)} compounds kept at Tanimoto {SIMILARITY}, from {len(unique)}")
print(f"median rmsd: {unique.rmsd.median():.3f} before, {leaders.rmsd.median():.3f} kept")

Save the non-redundant list, with the same two columns and sorted by MolPort ID. A second file holds
the same compounds, in the same order, with only the `smiles` column: this is the input file for the
Ersilia Model Hub.

In [ ]:
final = leaders[["smiles", "molport_id"]].sort_values("molport_id")
final.to_csv(BIG / "pharmit_hits_molport_nonredundant.csv", index=False)
final[["smiles"]].to_csv(BIG / "pharmit_hits_nonredundant_smiles.csv", index=False)
print(BIG / "pharmit_hits_molport_nonredundant.csv")
print(BIG / "pharmit_hits_nonredundant_smiles.csv")
final.head()

## Summary

- Read every Pharmit pose, and collapsed conformers and stereoisomers into one row per MolPort compound.
- Merged salt, charge and tautomer forms of the same molecule.
- Removed close analogues at Tanimoto 0.7 with a leader picker ordered by pharmacophore fit, so each
  family is represented by its best-fitting member (`pharmit_hits_molport_nonredundant.csv`, plus
  `pharmit_hits_nonredundant_smiles.csv` with the SMILES only).

**Next:** run `pharmit_hits_nonredundant_smiles.csv` through Ersilia Model Hub models (for example for
ADMET properties) to choose which compounds to order.